
# <span style='color:#e91e63'>EquiPhysics Gait Lesson 1: EquiPro Walk and Trot Data with a Phase-Oscillator Model</span>

This notebook is a first bridge between **real horse gait data** and a **mathematical model of quadruped locomotion**.

We will:

1. import stride-by-stride EquiPro data,
2. select one **walk** segment and one **trot** segment,
3. normalize time so that each stride runs from **0 to 1**,
4. reorder the limbs to match the convention:
   **LH, RH, LF, RF**,
5. plot the observed footfall timing, and
6. compare the data to a **reduced coupled phase-oscillator CPG model**.

For this lesson we use:

- **walk segment = 1**
- **trot segment = 2**

These are both left-direction, soft-surface segments from the same EquiPro session.



## How to use this notebook

This is a **student-facing notebook with complete runnable code**.

A few notes:

- The EquiPro CSV stores **one row per detected stride**.
- We will use the EquiPro hoof-on timestamps to define the observed gait events.
- We will normalize each stride so the stride phase lives on the interval **[0, 1)**.
- We will use **LH (left hind)** as the reference limb, because that matches the paper convention.

The mathematical model in this notebook is a **reduced coupled phase-oscillator model**, which belongs to the central pattern generator (CPG) modeling family that people use for quadruped gait.


In [ ]:
# =========================
# Step 0 — Setup
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.size"] = 11 # Reverted font size

# @markdown ### File Selection Options
# @markdown If the default file is not found, you can upload it or specify a Google Drive path.
FILE_SOURCE_OPTION = "Upload from computer" # @param ["Use default (local/mnt/data)", "Upload from computer", "Specify Google Drive path"] {type:"string"}
GOOGLE_DRIVE_FILE_PATH = "" # @param {type:"string"}

CSV_FILENAME_TO_PROCESS = None # This will store the filename/path based on user selection
WALK_SEGMENT = 1
TROT_SEGMENT = 2

if FILE_SOURCE_OPTION == "Upload from computer":
    from google.colab import files
    print("Please upload your CSV file now:")
    uploaded = files.upload()
    if uploaded:
        CSV_FILENAME_TO_PROCESS = list(uploaded.keys())[0]
        print(f"Using uploaded file: {CSV_FILENAME_TO_PROCESS}")
    else:
        raise ValueError("No file uploaded. Please upload the CSV file or choose another option.")
elif FILE_SOURCE_OPTION == "Specify Google Drive path":
    from google.colab import drive
    print("Mounting Google Drive... Please authorize if prompted.")
    drive.mount('/content/drive')
    if GOOGLE_DRIVE_FILE_PATH:
        CSV_FILENAME_TO_PROCESS = GOOGLE_DRIVE_FILE_PATH
        print(f"Using Google Drive file: {CSV_FILENAME_TO_PROCESS}")
    else:
        raise ValueError("Google Drive path not specified. Please enter the full path to your CSV file in the 'GOOGLE_DRIVE_FILE_PATH' field.")
else: # "Use default (local/mnt/data)"
    CSV_FILENAME_TO_PROCESS = "20260119T121135-Duque-results.csv"
    print(f"Attempting to find default file: {CSV_FILENAME_TO_PROCESS}")

if CSV_FILENAME_TO_PROCESS is None:
    raise ValueError("CSV file path could not be determined. Please ensure a file is selected/provided.")

# Limb order matched to the Buono–Golubitsky paper:
# u1 -> LH, u2 -> RH, u3 -> LF, u4 -> RF
LIMB_ORDER = ["LH", "RH", "LF", "RF"]

# Target phase offsets from the paper, written in the same limb order.
# These are relative to LH = 0.
PAPER_PHASES = {
    "walk": {"LH": 0.00, "RH": 0.50, "LF": 0.25, "RF": 0.75},
    "trot": {"LH": 0.00, "RH": 0.50, "LF": 0.50, "RF": 0.00},
}

LIMB_COLORS = {
    "LH": "#2e7d32",  # green
    "RH": "#ef6c00",  # orange
    "LF": "#1565c0",  # blue
    "RF": "#c62828",  # red
}

def find_csv(filename):
    candidates = [
        Path(filename),
        Path("/mnt/data") / filename,
        Path.cwd() / filename,
    ]
    # If the user provided an absolute path (e.g., from Google Drive), prioritize it
    if Path(filename).is_absolute():
        candidates.insert(0, Path(filename))

    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {filename}. Put the CSV in the same folder as this notebook "
        "or update CSV_FILENAME in the setup cell or use the file selection options."
    )

csv_path = find_csv(CSV_FILENAME_TO_PROCESS)
print(f"Using CSV file: {csv_path}")



## Step 1 — Load the EquiPro data

According to the Equi-Pro manual, the CSV export is **stride-by-stride**, so each row represents one detected stride. The manual also defines the hoof-on timestamps `ts_hoof_on_LF/RF/LH/RH`, stride duration, duty factor, stance duration, swing duration, DAP, and LAP. That makes this file a very good starting point for stride-level timing analysis.

We will first load the full file and inspect the available segments.


In [ ]:
# @title

# =========================
# Step 1 — Load the full CSV
# =========================

df = pd.read_csv(csv_path)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

segment_summary = (
    df.groupby(["segment", "gait", "direction", "surface"])
      .size()
      .reset_index(name="n_strides")
      .sort_values("segment")
)

display(segment_summary)


## Step 2 — Helper functions

The next cell defines a few helper functions that we will use throughout the lesson.

The most important one is `add_normalized_phases()`. For each stride, it computes hoof-on phase values using

$$
\text{phase}_{\text{limb}} = \left( \frac{t_{\text{hoof-on, limb}} - t_{\text{hoof-on, LH}}}{\text{stride duration}} \right) \bmod 1.
$$

Because we use **LH as the reference limb**, the normalized LH phase is always 0.

In [3]:
# @title
# =========================
# Step 2 — Helper functions
# =========================

def select_segment(data, segment_number):
    segment_df = data.loc[data["segment"] == segment_number].copy()
    if segment_df.empty:
        raise ValueError(f"No rows found for segment {segment_number}.")
    return segment_df.reset_index(drop=True)

def add_normalized_phases(segment_df, reference_limb="LH", limb_order=LIMB_ORDER):
    out = segment_df.copy()
    ref_col = f"ts_hoof_on_{reference_limb}"
    for limb in limb_order:
        hoof_col = f"ts_hoof_on_{limb}"
        out[f"phase_{limb}"] = ((out[hoof_col] - out[ref_col]) / out["stride_dur"]) % 1.0
    return out

def center_phases(phases, center):
    phases = np.asarray(phases, dtype=float)
    return center + ((phases - center + 0.5) % 1.0 - 0.5)

def summarize_segment_phases(segment_df, gait_name, limb_order=LIMB_ORDER):
    rows = []
    for limb in limb_order:
        center = PAPER_PHASES[gait_name][limb]
        centered = center_phases(segment_df[f"phase_{limb}"], center)
        rows.append({
            "limb": limb,
            "paper_phase": center,
            "observed_mean_phase": centered.mean(),
            "observed_sd_phase": centered.std(ddof=1),
        })
    summary = pd.DataFrame(rows)
    summary["gait"] = gait_name
    return summary[["gait", "limb", "paper_phase", "observed_mean_phase", "observed_sd_phase"]]

def wrap_for_plot(phases, center):
    return center_phases(phases, center)

def plot_phase_raster(segment_df, gait_name, ax, title_suffix=""):
    n = len(segment_df)
    y = np.arange(1, n + 1)

    for limb in LIMB_ORDER:
        phases = wrap_for_plot(segment_df[f"phase_{limb}"], PAPER_PHASES[gait_name][limb])

        for shift in (-1, 0, 1):
            ax.scatter(
                phases + shift,
                y,
                s=12,
                alpha=0.6,
                color=LIMB_COLORS[limb],
                label=limb if shift == 0 else None,
            )

        ax.axvline(
            PAPER_PHASES[gait_name][limb],
            color=LIMB_COLORS[limb],
            linestyle="--",
            linewidth=1,
            alpha=0.7,
        )

    ax.set_xlim(-0.1, 1.1)
    ax.set_xlabel("Normalized hoof-on phase")
    ax.set_ylabel("Stride index")
    ax.set_title(f"{gait_name.capitalize()} footfall phases{title_suffix}")
    ax.set_xticks([0.0, 0.25, 0.5, 0.75, 1.0])
    ax.grid(alpha=0.25)

def plot_stride_metrics(segment_df, gait_name, ax1, ax2):
    stride_idx = np.arange(1, len(segment_df) + 1)

    ax1.plot(stride_idx, segment_df["stride_dur"], marker="o", markersize=2, linewidth=1.2)
    ax1.set_title(f"{gait_name.capitalize()} stride duration")
    ax1.set_xlabel("Stride index")
    ax1.set_ylabel("Stride duration (s)")
    ax1.grid(alpha=0.25)

    for limb in LIMB_ORDER:
        col_name = f"duty_factor_{limb}"
        if col_name in segment_df.columns:
            ax2.plot(stride_idx, segment_df[col_name], marker="o", markersize=2, linewidth=1.2, color=LIMB_COLORS[limb], label=limb)
        elif "duty_factor" in segment_df.columns and limb == "LH": # Fallback
            ax2.plot(stride_idx, segment_df["duty_factor"], marker="o", markersize=2, linewidth=1.2, color="gray", label="Overall")

    ax2.set_title(f"{gait_name.capitalize()} duty factor")
    ax2.set_xlabel("Stride index")
    ax2.set_ylabel("Duty factor (%)")
    ax2.grid(alpha=0.25)
    ax2.legend(loc="best")

def compare_phase_table(observed_summary, model_phase_dict):
    merged = observed_summary.copy()
    merged["model_phase"] = merged["limb"].map(model_phase_dict)

    diffs_data = []
    diffs_model = []
    for _, row in merged.iterrows():
        center = row["paper_phase"]
        data_diff = center_phases([row["observed_mean_phase"]], center)[0] - center
        model_diff = center_phases([row["model_phase"]], center)[0] - center
        diffs_data.append(data_diff)
        diffs_model.append(model_diff)

    merged["observed_minus_paper"] = diffs_data
    merged["model_minus_paper"] = diffs_model
    return merged


## Step 3 — Select the walk and trot sequences

We now extract the two segments we chose for this lesson:

- **walk = segment 1**
- **trot = segment 2**

Then we compute normalized hoof-on phases for all four limbs.


In [ ]:
# @title

# =========================
# Step 3 — Select the two teaching segments
# =========================

walk_df = add_normalized_phases(select_segment(df, WALK_SEGMENT), reference_limb="LH")
trot_df = add_normalized_phases(select_segment(df, TROT_SEGMENT), reference_limb="LH")

print("Walk segment metadata:")
display(walk_df[["segment", "gait", "direction", "surface"]].head(1))

print("Trot segment metadata:")
display(trot_df[["segment", "gait", "direction", "surface"]].head(1))

observed_walk_summary = summarize_segment_phases(walk_df, "walk")
observed_trot_summary = summarize_segment_phases(trot_df, "trot")

print("Observed phase summary for walk:")
display(observed_walk_summary)

print("Observed phase summary for trot:")
display(observed_trot_summary)

print("Average stride metrics:")
metric_summary = pd.DataFrame({
    "gait": ["walk", "trot"],
    "n_strides": [len(walk_df), len(trot_df)],
    "mean_stride_duration_s": [walk_df["stride_dur"].mean(), trot_df["stride_dur"].mean()],
    "sd_stride_duration_s": [walk_df["stride_dur"].std(ddof=1), trot_df["stride_dur"].std(ddof=1)],
    "mean_duty_factor_pct": [walk_df["duty_factor"].mean(), trot_df["duty_factor"].mean()],
    "sd_duty_factor_pct": [walk_df["duty_factor"].std(ddof=1), trot_df["duty_factor"].std(ddof=1)],
})
display(metric_summary)



## Step 4 — Plot the observed EquiPro gait timing

The next cell makes three kinds of plots:

1. a **footfall phase raster**, where each point is a hoof-on event placed on the normalized stride interval, and  
2. a **stride metric plot** for stride duration.
3. a **stride metric plot** for the duty factor of each leg during each stride.

The dashed vertical lines in the raster show the **phase values** for the ideal walk and trot patterns.


In [ ]:
# @title
# =========================
# Step 4 — Plot observed walk and trot timing
# =========================

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
plot_phase_raster(walk_df, "walk", axes[0], title_suffix=" (segment 1)")
plot_phase_raster(trot_df, "trot", axes[1], title_suffix=" (segment 2)")

handles, labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, labels, loc="upper right")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

# Walk on the left (column 0), Trot on the right (column 1)
# Top row (axes[0, :]) for Stride duration, Bottom row (axes[1, :]) for Duty factor
plot_stride_metrics(walk_df, "walk", axes[0, 0], axes[1, 0])
plot_stride_metrics(trot_df, "trot", axes[0, 1], axes[1, 1])

# Limit the duty factor graphs (bottom row) to the first 20 strides
axes[1, 0].set_xlim(0, 20)
axes[1, 1].set_xlim(0, 20)

plt.show()


### What to look for

For the **walk** segment, the paper predicts the order

**LH → LF → RH → RF**

with roughly quarter-cycle spacing.

For the **trot** segment, the paper predicts diagonal synchronization:

- **LH and RF** together,
- **RH and LF** together,

with about a half-cycle offset between the two diagonal pairs.

You should already be able to see that the data cluster near these idealized phase patterns, but the horse is not perfectly periodic. That stride-to-stride variation is part of what makes the data interesting.


## Step 5 — A reduced coupled phase-oscillator model

Now we introduce a mathematical model that people actually use in locomotion modeling: a **coupled phase-oscillator CPG model**.

Instead of trying to model every physiological detail, we assign one phase variable to each limb oscillator:

$$
\theta_{\mathrm{LH}}(t), \quad \theta_{\mathrm{RH}}(t), \quad
\theta_{\mathrm{LF}}(t), \quad \theta_{\mathrm{RF}}(t).
$$

A simple reduced model is

$$
\dot{\theta}_i
=
\omega
+
\frac{K}{N-1}
\sum_{j \neq i}
\sin\Big((\theta_j-\theta_i)-(\phi_j^*-\phi_i^*)\Big),
$$

where

- $\omega$ is the natural stepping frequency,
- $K$ is the coupling strength,
- $N=4$ is the number of limb oscillators,
- $\phi_i^*$ are the **target phase lags** for a chosen gait.

This is a real CPG-style modeling idea: the oscillators interact so that their **relative phases lock** into a stable gait pattern.

For this lesson:

- the **walk** target phases are $[0,\; 1/2,\; 1/4,\; 3/4]$ in the order **LH, RH, LF, RF**,
- the **trot** target phases are $[0,\; 1/2,\; 1/2,\; 0]$.


In [ ]:
# @title

# =========================
# Step 5 — Simulate a reduced coupled phase-oscillator model
# =========================

def simulate_phase_cpg(target_phase_dict, n_cycles=12, dt=0.002, K=6.0, omega=2*np.pi, seed=7):
    # Simulate a reduced 4-limb phase-oscillator model.
    rng = np.random.default_rng(seed)
    limbs = LIMB_ORDER.copy()
    n = len(limbs)

    phi_star = np.array([target_phase_dict[limb] for limb in limbs], dtype=float) * 2*np.pi
    theta = rng.uniform(0, 2*np.pi, size=n)

    times = np.arange(0.0, n_cycles + dt, dt)
    theta_hist = np.zeros((len(times), n))
    theta_hist[0, :] = theta

    for k in range(1, len(times)):
        coupling = np.zeros(n)
        for i in range(n):
            phase_diffs = theta - theta[i]
            target_diffs = phi_star - phi_star[i]
            coupling[i] = np.sum(np.sin(phase_diffs - target_diffs))

        theta = theta + dt * (omega + (K / (n - 1)) * coupling)
        theta_hist[k, :] = theta

    return times, theta_hist

def locked_relative_phases(theta_hist):
    # Compute the final relative phases with LH used as the zero-phase reference.
    rel = ((theta_hist[-1, :] - theta_hist[-1, 0]) / (2*np.pi)) % 1.0
    return dict(zip(LIMB_ORDER, rel))

def relative_phase_history(theta_hist, gait_name):
    # Return phase histories wrapped around the paper's target phases.
    rel = ((theta_hist - theta_hist[:, [0]]) / (2*np.pi)) % 1.0
    centered = np.zeros_like(rel)
    for j, limb in enumerate(LIMB_ORDER):
        center = PAPER_PHASES[gait_name][limb]
        centered[:, j] = center_phases(rel[:, j], center)
    return centered

def model_activation_curves(locked_phase_dict, n_points=500):
    # Build one normalized stride of smooth oscillator outputs.
    tau = np.linspace(0, 1, n_points)
    curves = {}
    for limb in LIMB_ORDER:
        phase = locked_phase_dict[limb]
        curves[limb] = np.cos(2*np.pi * (tau - phase))
    return tau, curves

walk_times, walk_theta = simulate_phase_cpg(PAPER_PHASES["walk"], seed=5)
trot_times, trot_theta = simulate_phase_cpg(PAPER_PHASES["trot"], seed=12)

walk_model_phases = locked_relative_phases(walk_theta)
trot_model_phases = locked_relative_phases(trot_theta)

print("Locked model phases for walk:")
display(pd.DataFrame({"limb": LIMB_ORDER, "phase": [walk_model_phases[l] for l in LIMB_ORDER]}))

print("Locked model phases for trot:")
display(pd.DataFrame({"limb": LIMB_ORDER, "phase": [trot_model_phases[l] for l in LIMB_ORDER]}))



## Step 6 — Plot the model and compare it to the data

We will make three comparisons for each gait:

1. **phase-locking over time** in the reduced CPG model,
2. one normalized stride of **oscillator output curves**, and
3. **observed vs modeled hoof-on phases**.

The oscillator curves are from the model and are not direct measurements from EquiPro. They are a convenient way to visualize the fact that each limb oscillator is periodic and phase-shifted relative to the others.


In [ ]:
# @title
# =========================
# Step 6 — Plot model convergence, model traces, and observed-vs-modeled phases
# =========================

def plot_model_results(times, theta_hist, gait_name, segment_df, model_phase_dict):
    rel_hist = relative_phase_history(theta_hist, gait_name)
    tau, curves = model_activation_curves(model_phase_dict)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), constrained_layout=True)

    for j, limb in enumerate(LIMB_ORDER):
        axes[0].plot(times, rel_hist[:, j], label=limb, color=LIMB_COLORS[limb], linewidth=1.5)
        axes[0].axhline(PAPER_PHASES[gait_name][limb], color=LIMB_COLORS[limb], linestyle="--", alpha=0.5)
    axes[0].set_title(f"{gait_name.capitalize()} model: phase locking")
    axes[0].set_xlabel("Normalized time (strides)")
    axes[0].set_ylabel("Relative phase")
    axes[0].set_ylim(-0.1, 1.1)
    axes[0].grid(alpha=0.25)
    axes[0].legend(loc="best")

    for limb in LIMB_ORDER:
        axes[1].plot(tau, curves[limb], label=limb, color=LIMB_COLORS[limb], linewidth=2)
        model_phase = center_phases([model_phase_dict[limb]], PAPER_PHASES[gait_name][limb])[0]
        axes[1].axvline(model_phase, color=LIMB_COLORS[limb], linestyle="--", alpha=0.5)
    axes[1].set_title(f"{gait_name.capitalize()} model: one normalized cycle")
    axes[1].set_xlabel("Normalized stride phase")
    axes[1].set_ylabel("Oscillator output")
    axes[1].set_xlim(0, 1)
    axes[1].grid(alpha=0.25)

    for idx, limb in enumerate(LIMB_ORDER):
        # Get the raw phases from the dataframe and center them around the expected paper phase
        paper_phase = PAPER_PHASES[gait_name][limb]
        raw_phases = segment_df[f"phase_{limb}"]
        centered_phases = center_phases(raw_phases, paper_phase)

        # Create the box plot for the observed data
        axes[2].boxplot(
            centered_phases,
            positions=[idx + 0.15],
            vert=False,
            widths=0.35,
            patch_artist=True,
            boxprops=dict(facecolor=LIMB_COLORS[limb], color=LIMB_COLORS[limb], alpha=0.5),
            capprops=dict(color=LIMB_COLORS[limb]),
            whiskerprops=dict(color=LIMB_COLORS[limb]),
            flierprops=dict(marker='o', color=LIMB_COLORS[limb], markeredgecolor=LIMB_COLORS[limb], markersize=4, alpha=0.4),
            medianprops=dict(color=LIMB_COLORS[limb], linewidth=2)
        )

        # Plot the model's locked phase
        model_phase = center_phases([model_phase_dict[limb]], paper_phase)[0]
        axes[2].scatter(
            model_phase, idx - 0.15,
            marker="s", s=60, color=LIMB_COLORS[limb], zorder=3
        )

        # Reference line for ideal phase
        axes[2].axvline(paper_phase, color=LIMB_COLORS[limb], linestyle="--", alpha=0.35)

    axes[2].set_yticks(np.arange(len(LIMB_ORDER)))
    axes[2].set_yticklabels(LIMB_ORDER)
    axes[2].set_xlim(-0.1, 1.1)
    axes[2].set_xlabel("Normalized phase")
    axes[2].set_title(f"{gait_name.capitalize()}: observed vs modeled phase")
    axes[2].grid(alpha=0.25)

    plt.show()

# We pass walk_df and trot_df here instead of observed_walk_summary / observed_trot_summary
plot_model_results(walk_times, walk_theta, "walk", walk_df, walk_model_phases)
plot_model_results(trot_times, trot_theta, "trot", trot_df, trot_model_phases)


## Step 7 — Compare the phase values numerically

The next cell builds tables that compare:

- the **paper phase value**,
- the **observed EquiPro mean phase**,
- the **model phase** after locking,
- and the difference from the ideal phase.

Small differences mean the data or model are close to the idealized gait timing pattern.


In [ ]:
# @title

# =========================
# Step 7 — Numerical comparison tables
# =========================

walk_compare = compare_phase_table(observed_walk_summary, walk_model_phases)
trot_compare = compare_phase_table(observed_trot_summary, trot_model_phases)

print("Walk comparison table:")
display(walk_compare)

print("Trot comparison table:")
display(trot_compare)



## Step 9 — What did we learn?

1. We started with **real EquiPro stride data**.
2. We extracted hoof-on timing and normalized each stride to **phase on [0,1)**.
3. We matched the limb order to the convention **LH, RH, LF, RF**.
4. We saw that the measured walk and trot segments cluster near the expected phase patterns.
5. We simulated a **reduced coupled phase-oscillator model** and saw that it phase-locks to the desired gait.

### Key idea

A gait can be viewed as a **stable phase relationship among coupled oscillators**.

### Important limitation

The EquiPro CSV in this notebook is a **stride summary file**, not a full continuous time-series export. So our comparison is mainly about **event timing and relative phase**, not about continuous limb waveforms.

---

### Optional discussion questions

1. Why is it useful to normalize time by stride duration?
2. Where do the real data deviate from the idealized model, and why might that happen?
3. How would you modify the model if you wanted to include asymmetry?
